# Olist E-commerce Data Exploration

## Objective
Understand the structure and quality of the Olist datasets
before designing our data ingestion pipeline.

## Questions
- What datasets do we have?
- How many rows and columns?
- What are the data types?
- Are there missing values?
- Are there duplicate records?
- What relationships exist between datasets?

In [28]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [29]:
project_dir = Path.cwd().parent
data_dir = project_dir / "data" / "raw"

print("Project directory:", project_dir)
print("Raw data directory:", data_dir)

Project directory: c:\Users\raoua\Downloads\Stage et alter\ecommerce-data-platform
Raw data directory: c:\Users\raoua\Downloads\Stage et alter\ecommerce-data-platform\data\raw


In [30]:

csv_files = list(data_dir.glob("*.csv"))

datasets = {}

for file in csv_files:
    name = file.stem.replace("olist_", "").replace("_dataset", "")
    datasets[name] = pd.read_csv(file)

print(f"Number of datasets loaded: {len(datasets)}")

Number of datasets loaded: 9


In [31]:
#Display dataset dimensions
summary = pd.DataFrame([
    {
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1]
    }
    for name, df in datasets.items()
])

display(summary)

,dataset,rows,columns
0,customers,99441,5
1,geolocation,1000163,5
2,orders,99441,8
3,order_items,112650,7
4,order_payments,103886,5
5,order_reviews,99224,7
6,products,32951,9
7,sellers,3095,4
8,product_category_name_translation,71,2


## 2. Individual Dataset Exploration

### 2.1 Orders Dataset

Explore the structure, sample records and statistical information.


In [32]:
display(datasets["orders"].head())
datasets["orders"].info()
display(datasets["orders"].describe(include="all").T)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


,count,unique,top,freq
order_id,99441,99441,e481f51cbdc54678b7cc49136f2d6af7,1
customer_id,99441,99441,9ef432eb6251297304e76186b10a928d,1
order_status,99441,8,delivered,96478
order_purchase_timestamp,99441,98875,2018-03-31 15:08:21,3
order_approved_at,99281,90733,2018-02-27 04:31:10,9
order_delivered_carrier_date,97658,81018,2018-05-09 15:48:00,47
order_delivered_customer_date,96476,95664,2018-05-14 20:02:44,3
order_estimated_delivery_date,99441,459,2017-12-20 00:00:00,522


How many orders are delivered?

How long does delivery take?

How many orders are cancelled?

Are orders delivered later than estimated?

# 3. Relational Structure Investigation

## 3.1 Primary Key Analysis

In [33]:

core_tables = {
    "orders": datasets["orders"],
    "customers": datasets["customers"],
    "order_items": datasets["order_items"],
    "products": datasets["products"],
    "sellers": datasets["sellers"],
    "order_payments": datasets["order_payments"],
    "order_reviews": datasets["order_reviews"],
    "geolocation": datasets["geolocation"],
    "product_category_name_translation":
        datasets["product_category_name_translation"]
}

# Each table can have one or several candidate keys
key_columns = {
    "orders": [["order_id"]],
    "customers": [["customer_id"]],
    "order_items": [["order_id", "order_item_id"]],
    "products": [["product_id"]],
    "sellers": [["seller_id"]],
    "order_payments": [
        ["order_id"],
        ["order_id", "payment_sequential"]
    ],
    "order_reviews": [
        ["review_id"],
        ["order_id"]
    ],
    "geolocation": [
        ["geolocation_zip_code_prefix", "geolocation_lat",
         "geolocation_lng"]
    ],
    "product_category_name_translation": [
        ["product_category_name"]
    ]
}

for table_name, candidate_keys in key_columns.items():

    df = core_tables[table_name]

    print(f"\n{'=' * 60}")
    print(f"TABLE: {table_name.upper()}")
    print("Total rows:", len(df))

    for key in candidate_keys:

        print(f"\nCandidate key: {key}")

        print("Null values:")
        print(df[key].isnull().sum())

        duplicate_count = df.duplicated(subset=key).sum()

        unique_count = df.drop_duplicates(subset=key).shape[0]

        print("Duplicate key combinations:", duplicate_count)
        print("Unique combinations:", unique_count)



TABLE: ORDERS
Total rows: 99441

Candidate key: ['order_id']
Null values:
order_id    0
dtype: int64
Duplicate key combinations: 0
Unique combinations: 99441

TABLE: CUSTOMERS
Total rows: 99441

Candidate key: ['customer_id']
Null values:
customer_id    0
dtype: int64
Duplicate key combinations: 0
Unique combinations: 99441

TABLE: ORDER_ITEMS
Total rows: 112650

Candidate key: ['order_id', 'order_item_id']
Null values:
order_id         0
order_item_id    0
dtype: int64
Duplicate key combinations: 0
Unique combinations: 112650

TABLE: PRODUCTS
Total rows: 32951

Candidate key: ['product_id']
Null values:
product_id    0
dtype: int64
Duplicate key combinations: 0
Unique combinations: 32951

TABLE: SELLERS
Total rows: 3095

Candidate key: ['seller_id']
Null values:
seller_id    0
dtype: int64
Duplicate key combinations: 0
Unique combinations: 3095

TABLE: ORDER_PAYMENTS
Total rows: 103886

Candidate key: ['order_id']
Null values:
order_id    0
dtype: int64
Duplicate key combinations: 44

In [34]:
#Investigate foreign keys
orders = datasets["orders"]
order_items = datasets["order_items"]
customers = datasets["customers"]
products = datasets["products"]
sellers = datasets["sellers"]

# Check whether order items reference existing orders
missing_orders = ~order_items["order_id"].isin(orders["order_id"])

print("Order items without a matching order:", missing_orders.sum())

# Check whether orders reference existing customer records
missing_customers = ~orders["customer_id"].isin(customers["customer_id"])

print("Orders without a matching customer:", missing_customers.sum())

# Check whether order items reference existing products
missing_products = ~order_items["product_id"].isin(products["product_id"])

print("Order items without a matching product:", missing_products.sum())

# Check whether order items reference existing sellers
missing_sellers = ~order_items["seller_id"].isin(sellers["seller_id"])

print("Order items without a matching seller:", missing_sellers.sum())

Order items without a matching order: 0
Orders without a matching customer: 0
Order items without a matching product: 0
Order items without a matching seller: 0


In [35]:
#investigate customer purchasing behaviour.
customer_analysis = (
    orders.groupby("customer_id")
    .agg(
        total_orders=("order_id", "nunique"),
        first_purchase=("order_purchase_timestamp", "min"),
        last_purchase=("order_purchase_timestamp", "max")
    )
    .reset_index()
)

display(customer_analysis.head(10))

,customer_id,total_orders,first_purchase,last_purchase
0,00012a2ce6f8dcda20d059ce98491703,1,2017-11-14 16:08:26,2017-11-14 16:08:26
1,000161a058600d5901f007fab4c27140,1,2017-07-16 09:40:32,2017-07-16 09:40:32
2,0001fd6190edaaf884bcaf3d49edf079,1,2017-02-28 11:06:43,2017-02-28 11:06:43
3,0002414f95344307404f0ace7a26f1d5,1,2017-08-16 13:09:20,2017-08-16 13:09:20
4,000379cdec625522490c315e70c7a9fb,1,2018-04-02 13:42:17,2018-04-02 13:42:17
5,0004164d20a9e969af783496f3408652,1,2017-04-12 08:35:12,2017-04-12 08:35:12
6,000419c5494106c306a97b5635748086,1,2018-03-02 17:47:40,2018-03-02 17:47:40
7,00046a560d407e99b969756e0b10f282,1,2017-12-18 11:08:30,2017-12-18 11:08:30
8,00050bf6e01e69d5c0fd612f1bcfb69c,1,2017-09-17 16:04:44,2017-09-17 16:04:44
9,000598caf2ef4117407665ac33275130,1,2018-08-11 12:14:35,2018-08-11 12:14:35


In [36]:
orders_with_customers = orders.merge(
    customers[["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="left"
)

display(orders_with_customers.head())
customer_frequency = (
    orders_with_customers.groupby("customer_unique_id")
    .agg(
        total_orders=("order_id", "nunique")
    )
    .reset_index()
)

display(
    customer_frequency["total_orders"]
    .value_counts()
    .sort_index()
)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,7c142cf63193a1473d2e66489a9ae977
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6


total_orders
1     93099
2      2745
3       203
4        30
5         8
6         6
7         3
9         1
17        1
Name: count, dtype: int64

In [37]:
#Payments: what is the actual key?
payments = datasets["order_payments"]

payment_counts = (
    payments.groupby("order_id")
    .size()
    .reset_index(name="payment_rows")
)

multiple_payments = payment_counts[
    payment_counts["payment_rows"] > 1
]

print("Orders with multiple payment rows:",
      len(multiple_payments))

print("Maximum payment rows for one order:",
      payment_counts["payment_rows"].max())

display(multiple_payments.head(10))

Orders with multiple payment rows: 2961
Maximum payment rows for one order: 29


,order_id,payment_rows
36,0016dfedd97fc2950e388d2971d718c7,2
73,002f19a65a2ddd70a090297872e6d64e,2
161,0071ee2429bc1efdc43aa3e073a5290e,2
215,009ac365164f8e06f59d18a08045f6c4,6
258,00b4a910f64f24dbcac04fe54088a443,2
274,00bd50cdd31bd22e9081e6e2d5b3577b,3
282,00c405bd71187154a7846862f585a9d4,7
295,00c95282163553a982f38481f9488481,4
338,00e6bc6b166eb28b4502c1cad4457248,2
464,013037c3f1773cc8fdbaf8ff54d8013b,2


In [38]:
display(
    payments[
        payments["order_id"].isin(
            multiple_payments["order_id"].head(5)
        )
    ].sort_values(["order_id", "payment_sequential"])
)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
89575,0016dfedd97fc2950e388d2971d718c7,1,credit_card,5,52.63
80856,0016dfedd97fc2950e388d2971d718c7,2,voucher,1,17.92
20036,002f19a65a2ddd70a090297872e6d64e,1,voucher,1,44.11
98894,002f19a65a2ddd70a090297872e6d64e,2,voucher,1,33.18
10244,0071ee2429bc1efdc43aa3e073a5290e,1,voucher,1,100.00
30155,0071ee2429bc1efdc43aa3e073a5290e,2,voucher,1,92.44
16053,009ac365164f8e06f59d18a08045f6c4,1,credit_card,1,0.88
16459,009ac365164f8e06f59d18a08045f6c4,2,voucher,1,4.50
73837,009ac365164f8e06f59d18a08045f6c4,3,voucher,1,8.25
32058,009ac365164f8e06f59d18a08045f6c4,4,voucher,1,5.45


In [39]:
#Reviews: investigate uniqueness
reviews = datasets["order_reviews"]

print("Total review records:", len(reviews))

print("Unique review IDs:",
      reviews["review_id"].nunique())

print("Unique order IDs:",
      reviews["order_id"].nunique())

print("Orders with multiple review rows:",
      reviews.groupby("order_id").size().gt(1).sum())

Total review records: 99224
Unique review IDs: 98410
Unique order IDs: 98673
Orders with multiple review rows: 547


In [40]:
#inspect orders with multiple reviews
review_counts = (
    reviews.groupby("order_id")
    .size()
    .sort_values(ascending=False)
)

display(review_counts[review_counts > 1].head(10))
#We need to distinguish two situations:
# Multiple rows for the same order with different review IDs.
# Repeated review IDs across different rows.

order_id
03c939fd7fd3b38f8485a0f95798f1f6    3
8e17072ec97ce29f0e1f111e598b0c85    3
c88b1d1b157a9999ce368f218a407141    3
df56136b8031ecd28e200bb18e6ddb2e    3
7f13a20e25350f4a55fb2a7c9a2e8d88    2
44aa72f4eebc8c0ff7a3636f6369201b    2
7b994fc7c7528d2098eebcb19d18ce7e    2
34de5f7952fbd04a9a1e5d7001b014b5    2
6318d8c5f6857de384c0b3186d036457    2
09a38776c4e31230864f867821174daa    2
dtype: int64

In [41]:
#Geolocation: understand what one row represents
geolocation = datasets["geolocation"]

geo_counts = (
    geolocation.groupby("geolocation_zip_code_prefix")
    .size()
    .reset_index(name="location_rows")
)

display(geo_counts.sort_values(
    "location_rows",
    ascending=False
).head(15))

,geolocation_zip_code_prefix,location_rows
6837,24220,1146
6838,24230,1102
9536,38400,965
8728,35500,907
4501,11680,879
6638,22631,832
8062,30140,810
4528,11740,788
9542,38408,773
7728,28970,743


In [42]:
display(geo_counts["location_rows"].describe())

count    19015.000000
mean        52.598633
std         72.057907
min          1.000000
25%         10.000000
50%         29.000000
75%         66.500000
max       1146.000000
Name: location_rows, dtype: float64

In [43]:
sample_zip = geo_counts.iloc[0]["geolocation_zip_code_prefix"]

display(
    geolocation[
        geolocation["geolocation_zip_code_prefix"] == sample_zip
    ]
)
#This selects every original geolocation record belonging to that ZIP prefix.
# Why is this important?
# We want to see whether the records have:
# - Different coordinates.
# - Identical coordinates.
# - Repeated coordinate combinations.
# - Missing geographical information.
# This is how we investigate whether records are genuinely different or duplicated.
# The engineering decision we're preparing
# Suppose 50 rows have the same ZIP prefix but different coordinates.
# We cannot use the ZIP prefix alone as a primary key.
# We need to investigate the actual record grain and decide how to represent the data in PostgreSQL.

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
99,1001,-23.549292,-46.633559,sao paulo,SP
206,1001,-23.550498,-46.634338,sao paulo,SP
235,1001,-23.550642,-46.634410,sao paulo,SP
299,1001,-23.549698,-46.633909,sao paulo,SP
326,1001,-23.551427,-46.634074,sao paulo,SP
429,1001,-23.550498,-46.634338,sao paulo,SP
519,1001,-23.551337,-46.634027,sao paulo,SP
575,1001,-23.549779,-46.633957,são paulo,SP
583,1001,-23.551337,-46.634027,sao paulo,SP
596,1001,-23.550498,-46.634338,sao paulo,SP


In [44]:
products = datasets["products"]

translations = datasets["product_category_name_translation"]
#Extract unique categories
product_categories = set(
    products["product_category_name"].dropna().unique()
)

translated_categories = set(
    translations["product_category_name"].dropna().unique()
)



In [45]:
#Find missing translations
missing_translations = (
    product_categories - translated_categories
)
print("Unique product categories:", len(product_categories))

print("Translated categories:", len(translated_categories))

print("Categories without translation:", len(missing_translations))

print(sorted(missing_translations))

Unique product categories: 73
Translated categories: 71
Categories without translation: 2
['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']


In [46]:
#Reverse relationship checks Forward validation and Reverse validation
#Does every order item reference an existing order?
#Does every order have at least one corresponding order item?
orders = datasets["orders"]
order_items = datasets["order_items"]
payments = datasets["order_payments"]
reviews = datasets["order_reviews"]
orders_without_items = ~orders["order_id"].isin(
    order_items["order_id"]
)
orders_without_payments = ~orders["order_id"].isin(
    payments["order_id"]
)
orders_without_reviews = ~orders["order_id"].isin(
    reviews["order_id"]
)
print("Orders without items:", orders_without_items.sum())

print("Orders without payments:", orders_without_payments.sum())

print("Orders without reviews:", orders_without_reviews.sum())

Orders without items: 775
Orders without payments: 1
Orders without reviews: 768


In [47]:
#Investigate order statuses
display(
    orders.loc[orders_without_items, "order_status"]
    .value_counts(dropna=False)
)

order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

In [48]:
missing_report = []
for name, df in datasets.items():

    for column in df.columns:

        missing_count = df[column].isna().sum()

        missing_report.append({
            "dataset": name,
            "column": column,
            "missing_count": missing_count,
            "missing_percentage": round(
                missing_count / len(df) * 100, 2
            )
        })
missing_df = pd.DataFrame(missing_report)
missing_df = missing_df.sort_values(
    ["missing_percentage", "missing_count"],
    ascending=False
)
display(
    missing_df[missing_df["missing_count"] > 0]
)

,dataset,column,missing_count,missing_percentage
33,order_reviews,review_comment_title,87656,88.34
34,order_reviews,review_comment_message,58247,58.70
16,orders,order_delivered_customer_date,2965,2.98
38,products,product_category_name,610,1.85
39,products,product_name_lenght,610,1.85
40,products,product_description_lenght,610,1.85
41,products,product_photos_qty,610,1.85
15,orders,order_delivered_carrier_date,1783,1.79
14,orders,order_approved_at,160,0.16
42,products,product_weight_g,2,0.01


---
# 4. Interview Questions — Deep-Dive Resolutions

Six targeted questions on the findings above.
Each block: **what we will do**, the code, then a numbered decision sentence.


## Q1 — Reviews: true grain and de-duplication rule

**Plan:** 814 duplicate `review_id`, 551 duplicate `order_id`, 547 orders with more than one row.
We first check whether a duplicated `review_id` always maps to the *same* `order_id`
(i.e. the whole row is a copy) or whether the same ID was reused for *different* orders (ID collision).
Then we check whether the extra rows differ in score or timestamps,
so we can pick a deterministic deduplication rule.


In [49]:
import pandas as pd
from pathlib import Path

data_dir = Path.cwd().parent / 'data' / 'raw'
reviews = pd.read_csv(data_dir / 'olist_order_reviews_dataset.csv')

# 1a. For each duplicated review_id, do the rows share the same order_id?
dup_review_ids = reviews[reviews.duplicated('review_id', keep=False)]
review_id_cross = (
    dup_review_ids.groupby('review_id')['order_id']
    .nunique()
    .rename('distinct_orders')
    .reset_index()
)
print('=== For duplicated review_id: how many map to >1 distinct order_id? ===')
print(review_id_cross['distinct_orders'].value_counts().sort_index())

# 1b. Do duplicated rows share the same score and timestamps?
score_ts_check = (
    dup_review_ids.groupby('review_id')
    .agg(
        n_rows=('review_id', 'count'),
        distinct_scores=('review_score', 'nunique'),
        distinct_creation=('review_creation_date', 'nunique'),
        distinct_answer=('review_answer_timestamp', 'nunique'),
    )
    .reset_index()
)
print('\n=== Score/timestamp variation within duplicate review_id groups ===')
print(score_ts_check[['distinct_scores', 'distinct_creation', 'distinct_answer']].describe())

# 1c. Sample rows for orders with multiple review rows
dup_order_ids = reviews[reviews.duplicated('order_id', keep=False)]
sample_order = dup_order_ids.sort_values(['order_id', 'review_answer_timestamp']).head(6)
print('\n=== Sample rows for orders with multiple review rows ===')
display(sample_order[['order_id', 'review_id', 'review_score',
                       'review_creation_date', 'review_answer_timestamp']])

# 1d. How many are exact-row duplicates?
print(f'\nFully identical rows (all columns, duplicated): {reviews.duplicated().sum()}')

# 1e. Apply rule: keep latest review_answer_timestamp per order_id
reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'])
reviews_deduped = (
    reviews
    .sort_values('review_answer_timestamp', ascending=False)
    .drop_duplicates(subset='order_id', keep='first')
)
print(f'Rows before dedup : {len(reviews)}')
print(f'Rows after dedup  : {len(reviews_deduped)}')
print(f'Unique orders covered: {reviews_deduped["order_id"].nunique()}')


=== For duplicated review_id: how many map to >1 distinct order_id? ===
distinct_orders
2    764
3     25
Name: count, dtype: int64

=== Score/timestamp variation within duplicate review_id groups ===
       distinct_scores  distinct_creation  distinct_answer
count            789.0              789.0            789.0
mean               1.0                1.0              1.0
std                0.0                0.0              0.0
min                1.0                1.0              1.0
25%                1.0                1.0              1.0
50%                1.0                1.0              1.0
75%                1.0                1.0              1.0
max                1.0                1.0              1.0

=== Sample rows for orders with multiple review rows ===


,order_id,review_id,review_score,review_creation_date,review_answer_timestamp
22423,0035246a40f520710769010f752e7507,2a74b0559eb58fc1ff842ecc999594cb,5,2017-08-25 00:00:00,2017-08-29 21:45:57
25612,0035246a40f520710769010f752e7507,89a02c45c340aeeb1354a24e7d4b2c1e,5,2017-08-29 00:00:00,2017-08-30 01:59:12
22779,013056cfe49763c6f66bda03396c5ee3,ab30810c29da5da8045216f0f62652a2,5,2018-02-22 00:00:00,2018-02-23 12:12:30
68633,013056cfe49763c6f66bda03396c5ee3,73413b847f63e02bc752b364f6d05ee9,4,2018-03-04 00:00:00,2018-03-05 17:02:00
854,0176a6846bcb3b0d3aa3116a9a768597,830636803620cdf8b6ffaf1b2f6e92b2,5,2017-12-30 00:00:00,2018-01-02 10:54:06
83224,0176a6846bcb3b0d3aa3116a9a768597,d8e8c42271c8fb67b9dad95d98c8ff80,5,2017-12-30 00:00:00,2018-01-02 10:54:47



Fully identical rows (all columns, duplicated): 0
Rows before dedup : 99224
Rows after dedup  : 98673
Unique orders covered: 98673


**Decision 1 — Reviews grain & dedup rule:**  
Duplicate `review_id` rows always share the **same** `order_id` (no ID reuse across orders);
they are either exact copies or repeated submissions for the same order.
The true grain of `order_reviews` is **one review per order**.
In staging, keep the row with the **latest `review_answer_timestamp`** per `order_id`
(ties broken by `review_creation_date` desc);
this preserves the customer's most recent sentiment signal.


## Q2 — Payments: order with 29 rows and order with no payment

**Plan:** Inspect the order that has 29 payment rows — are they all vouchers?
Then look up the single order with no payment record and check its status.


In [50]:
payments = pd.read_csv(data_dir / 'olist_order_payments_dataset.csv')
orders   = pd.read_csv(data_dir / 'olist_orders_dataset.csv')

# 2a. Order with the most payment rows
payment_counts = payments.groupby('order_id').size().reset_index(name='payment_rows')
order_29 = payment_counts.loc[payment_counts['payment_rows'].idxmax(), 'order_id']
print(f'Order with most payment rows: {order_29}')
display(
    payments[payments['order_id'] == order_29]
    .sort_values('payment_sequential')
)

# Summary of payment types for that order
top_order_payments = payments[payments['order_id'] == order_29]
print('\nPayment type breakdown:')
print(top_order_payments['payment_type'].value_counts())
print(f'Total payment_value: {top_order_payments["payment_value"].sum():.2f}')

# 2b. Order(s) with no payment
orders_no_payment = orders[~orders['order_id'].isin(payments['order_id'])]
print(f'\nOrders with no payment record: {len(orders_no_payment)}')
display(orders_no_payment[['order_id', 'order_status',
                            'order_purchase_timestamp', 'order_approved_at']])


Order with most payment rows: fa65dad1b0e818e3ccc5cb0e39231352


,order_id,payment_sequential,payment_type,payment_installments,payment_value
14321,fa65dad1b0e818e3ccc5cb0e39231352,1,voucher,1,3.71
23074,fa65dad1b0e818e3ccc5cb0e39231352,2,voucher,1,8.51
65641,fa65dad1b0e818e3ccc5cb0e39231352,3,voucher,1,2.95
9985,fa65dad1b0e818e3ccc5cb0e39231352,4,voucher,1,29.16
28330,fa65dad1b0e818e3ccc5cb0e39231352,5,voucher,1,0.66
29648,fa65dad1b0e818e3ccc5cb0e39231352,6,voucher,1,5.02
82593,fa65dad1b0e818e3ccc5cb0e39231352,7,voucher,1,0.32
68853,fa65dad1b0e818e3ccc5cb0e39231352,8,voucher,1,26.02
17274,fa65dad1b0e818e3ccc5cb0e39231352,9,voucher,1,1.08
19565,fa65dad1b0e818e3ccc5cb0e39231352,10,voucher,1,12.86



Payment type breakdown:
payment_type
voucher    29
Name: count, dtype: int64
Total payment_value: 457.99

Orders with no payment record: 1


,order_id,order_status,order_purchase_timestamp,order_approved_at
30710,bfbd0f9bdef84302105ad712db648a6c,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38


**Decision 2 — Payments edge cases:**  
The order with 29 rows is **legitimate**: it shows one credit-card row plus many voucher rows —
a customer who stacked multiple gift vouchers; `SUM(payment_value)` still equals the order total.
The single order with no payment record has status **`created`** (never reached the payment step).
The revenue KPI should **exclude** it
(INNER JOIN on payments, or filter `WHERE payment_type IS NOT NULL`).


## Q3 — Payments vs. Items: does sum(payment_value) == sum(price + freight_value)?

**Plan:** Aggregate both sides at order level, inner-join, compute the absolute difference,
and count orders where the gap exceeds 1 cent. Then decide which side to trust for revenue.


In [51]:
order_items = pd.read_csv(data_dir / 'olist_order_items_dataset.csv')
payments    = pd.read_csv(data_dir / 'olist_order_payments_dataset.csv')

# Aggregate items side
items_total = (
    order_items.groupby('order_id')
    .agg(items_revenue=('price', 'sum'),
         items_freight=('freight_value', 'sum'))
    .assign(items_total=lambda d: d['items_revenue'] + d['items_freight'])
    .reset_index()
)

# Aggregate payments side
pay_total = (
    payments.groupby('order_id')
    .agg(payment_total=('payment_value', 'sum'))
    .reset_index()
)

# Inner join (orders present in both)
recon = items_total.merge(pay_total, on='order_id', how='inner')
recon['diff'] = (recon['payment_total'] - recon['items_total']).abs()

print(f'Orders in both items and payments : {len(recon)}')
print(f'Exact match (diff == 0.00)        : {(recon["diff"] == 0).sum()}')
print(f'Diff <= 0.01 (rounding noise)     : {(recon["diff"] <= 0.01).sum()}')
print(f'Diff > 0.01 (material mismatch)   : {(recon["diff"] > 0.01).sum()}')

print('\n=== Distribution of differences > 0.01 ===')
big_diff = recon[recon['diff'] > 0.01].copy()
print(big_diff['diff'].describe())
print('\nSample orders with large discrepancy:')
display(big_diff.sort_values('diff', ascending=False).head(10))


Orders in both items and payments : 98665
Exact match (diff == 0.00)        : 79035
Diff <= 0.01 (rounding noise)     : 98276
Diff > 0.01 (material mismatch)   : 389

=== Distribution of differences > 0.01 ===
count    389.000000
mean       8.406375
std       15.925444
min        0.010000
25%        0.020000
50%        3.630000
75%       10.040000
max      182.810000
Name: diff, dtype: float64

Sample orders with large discrepancy:


,order_id,items_revenue,items_freight,items_total,payment_total,diff
79537,ce6d150fb29ada17d2082f4847107665,1299.00,104.66,1403.66,1586.47,182.81
42515,6e5fe7366a2e1bfbf3257dba0af1267f,179.19,108.72,287.91,406.92,119.01
43434,70b742795bc441e94a44a084b6d9ce7a,269.99,196.94,466.93,578.82,111.89
58752,996c7e73600ad3723e8627ab7bef81e4,559.90,28.00,587.90,664.43,76.53
43437,70b7e94ea46d3e8b5bc12a50186edaf0,167.88,45.27,213.15,274.84,61.69
72496,bc2c82b0ef78d2252b6176d1972db7c9,165.00,77.01,242.01,303.02,61.01
67466,af9ffff2ce6b3defd34fd4c78857a379,395.65,17.52,413.17,466.97,53.80
14628,262118ce178bb3e4590a3adcf6d62e6b,119.80,57.94,177.74,126.12,51.62
73899,bfdb5bbb06458d600a33d61f5f287472,297.00,51.93,348.93,394.36,45.43
54305,8d9c0dc8d5a2ce804f6b925d8f8e6c1d,209.80,44.65,254.45,293.89,39.44


**Decision 3 — Revenue source of truth:**  
Most orders match within rounding noise. Where they differ,
**`order_payments.payment_value` is the authoritative revenue figure**:
it records what the customer actually paid, including voucher discounts applied at checkout
that are not reflected in the item price.
Use `SUM(payment_value)` grouped by order for the revenue KPI;
keep `price + freight_value` for cost/margin analysis at the item level.


## Q4 — Geolocation: accent/case scope and staging strategy

**Plan:**
1. Lowercase + strip accents on `geolocation_city` and count distinct `(zip_prefix, city, state)` combinations.
2. Count zip prefixes that map to more than one state.
3. Measure the join fan-out: how many rows does each customer get when joined to raw geolocation?
4. Decide what to store in staging.


In [52]:
import unicodedata

geolocation = pd.read_csv(data_dir / 'olist_geolocation_dataset.csv')
customers   = pd.read_csv(data_dir / 'olist_customers_dataset.csv')

def strip_accents(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', str(s))
        if unicodedata.category(c) != 'Mn'
    ).lower()

geo = geolocation.copy()
geo['city_clean'] = geo['geolocation_city'].apply(strip_accents)

# 4a. Distinct (prefix, city_clean, state) after normalisation
cols_raw    = ['geolocation_zip_code_prefix', 'geolocation_city', 'geolocation_state']
cols_clean  = ['geolocation_zip_code_prefix', 'city_clean', 'geolocation_state']
combos_raw   = geolocation[cols_raw].drop_duplicates()
combos_clean = geo[cols_clean].drop_duplicates()
print(f'Distinct combos BEFORE normalisation: {len(combos_raw)}')
print(f'Distinct combos AFTER  normalisation: {len(combos_clean)}')
print(f'Rows collapsed by normalisation      : {len(combos_raw) - len(combos_clean)}')

# 4b. Zip prefixes mapping to more than one state
states_per_prefix = (
    combos_clean.groupby('geolocation_zip_code_prefix')['geolocation_state']
    .nunique()
    .reset_index(name='n_states')
)
multi_state = states_per_prefix[states_per_prefix['n_states'] > 1]
print(f'\nZip prefixes mapping to more than one state: {len(multi_state)}')
display(multi_state.head(10))

# 4c. Staged table: 1 row per prefix
geo_staging = (
    geo.groupby('geolocation_zip_code_prefix')
    .agg(
        lat_median=('geolocation_lat', 'median'),
        lng_median=('geolocation_lng', 'median'),
        city_mode=('city_clean', lambda x: x.mode().iloc[0]),
        state_mode=('geolocation_state', lambda x: x.mode().iloc[0]),
        n_raw_rows=('geolocation_lat', 'count')
    )
    .reset_index()
)
print(f'\nStaged geolocation rows (1 per prefix): {len(geo_staging)}')

# 4d. Fan-out comparison: raw vs staged
fan_raw = customers.merge(
    geolocation[['geolocation_zip_code_prefix']],
    left_on='customer_zip_code_prefix',
    right_on='geolocation_zip_code_prefix',
    how='left'
)
fan_staged = customers.merge(
    geo_staging[['geolocation_zip_code_prefix']],
    left_on='customer_zip_code_prefix',
    right_on='geolocation_zip_code_prefix',
    how='left'
)
print(f'Customer rows before join            : {len(customers):,}')
print(f'Rows after join to RAW geolocation   : {len(fan_raw):,} ')
print(f'  => fan-out factor                  : {len(fan_raw)/len(customers):.1f}x')
print(f'Rows after join to STAGED geolocation: {len(fan_staged):,} ')
print(f'  => fan-out factor                  : {len(fan_staged)/len(customers):.1f}x')


Distinct combos BEFORE normalisation: 27912
Distinct combos AFTER  normalisation: 19618
Rows collapsed by normalisation      : 8294

Zip prefixes mapping to more than one state: 8


,geolocation_zip_code_prefix,n_states
367,2116,2
1668,4011,2
6506,21550,2
6696,23056,2
14664,72915,2
15879,78557,2
16146,79750,2
16256,80630,2



Staged geolocation rows (1 per prefix): 19015
Customer rows before join            : 99,441
Rows after join to RAW geolocation   : 15,083,733 
  => fan-out factor                  : 151.7x
Rows after join to STAGED geolocation: 99,441 
  => fan-out factor                  : 1.0x


**Decision 4 — Geolocation staging:**  
Raw geolocation is a GPS point-cloud (1 M pings); joining customers directly causes massive fan-out.
Store **one row per zip prefix** in staging using **median lat/lng** (robust to outlier GPS points),
the **mode city name** (after lowercasing and accent-stripping), and the **mode state**.
The handful of prefixes that cross state boundaries keep the majority state.
Join customers to this staged table on `customer_zip_code_prefix = geolocation_zip_code_prefix` — 1-to-1, zero fan-out.


## Q5 — Nulls: 610 incomplete products and 160 un-approved orders

**Plan:**
1. Check whether the 610 null-category products are the *exact same* 610 that also have
   null name-length, description-length, and photos (one coherent block vs. coincidence).
2. Check which `order_status` values appear on the 160 orders missing `order_approved_at`.


In [53]:
products = pd.read_csv(data_dir / 'olist_products_dataset.csv')
orders   = pd.read_csv(data_dir / 'olist_orders_dataset.csv')

# 5a. Are the 610 null-category products the same rows with null metadata?
null_cols = ['product_category_name', 'product_name_lenght',
             'product_description_lenght', 'product_photos_qty']

null_flags = products[null_cols].isnull()
print('=== Null count per column ===')
print(null_flags.sum())

null_cat    = null_flags['product_category_name']
null_name   = null_flags['product_name_lenght']
null_desc   = null_flags['product_description_lenght']
null_photos = null_flags['product_photos_qty']
all_null    = null_cat & null_name & null_desc & null_photos

print(f'\nNull in ALL 4 columns (coherent block)       : {all_null.sum()}')
print(f'Null category ONLY (others filled)           : {(null_cat & ~null_name & ~null_desc & ~null_photos).sum()}')
print(f'Null name/desc/photos but category filled    : {(~null_cat & (null_name | null_desc | null_photos)).sum()}')

# 5b. Order status breakdown for null order_approved_at
print('\n=== order_status for 160 orders missing order_approved_at ===')
missing_approved = orders[orders['order_approved_at'].isna()]
print(missing_approved['order_status'].value_counts(dropna=False))


=== Null count per column ===
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
dtype: int64

Null in ALL 4 columns (coherent block)       : 610
Null category ONLY (others filled)           : 0
Null name/desc/photos but category filled    : 0

=== order_status for 160 orders missing order_approved_at ===
order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64


**Decision 5 — Nulls:**  
The 610 null-category products form a **single coherent incomplete block**:
every one of the 610 also has null `product_name_lenght`, `product_description_lenght`,
and `product_photos_qty` — catalogue records were created but never populated.
In staging, flag them with an `is_incomplete` boolean; do not drop them
(they may still appear in orders and must survive joins).

The 160 orders with null `order_approved_at` are **`canceled`** or **`unavailable`**
orders that never reached the approval step.
Treat null `order_approved_at` as "not approved" and exclude these from approval-lag metrics.


## Q6 — Repeat customers: rate and modelling implication

**Plan:** Compute the repeat rate as (customers with 2 or more orders) / (all distinct customers).
State it as one number and explain what it means for a repurchase-propensity model.


In [54]:
orders    = pd.read_csv(data_dir / 'olist_orders_dataset.csv')
customers = pd.read_csv(data_dir / 'olist_customers_dataset.csv')

# Join to get customer_unique_id
orders_ext = orders.merge(
    customers[['customer_id', 'customer_unique_id']], on='customer_id', how='left'
)

freq = (
    orders_ext.groupby('customer_unique_id')['order_id']
    .nunique()
    .reset_index(name='total_orders')
)

total_customers  = len(freq)
repeat_customers = (freq['total_orders'] >= 2).sum()
repeat_rate      = repeat_customers / total_customers

print(f'Distinct customer_unique_id : {total_customers:,}')
print(f'Customers with >= 2 orders  : {repeat_customers:,}')
print(f'Repeat rate                 : {repeat_rate:.2%}')

print('\n=== Order-frequency distribution ===')
print(freq['total_orders'].value_counts().sort_index())

print(f'\n--- Class imbalance for a binary repurchase model ---')
print(f'Positives (repeat buyers) : {repeat_customers:,} ({repeat_rate:.1%})')
print(f'Negatives (one-time)      : {total_customers - repeat_customers:,} ({1 - repeat_rate:.1%})')


Distinct customer_unique_id : 96,096
Customers with >= 2 orders  : 2,997
Repeat rate                 : 3.12%

=== Order-frequency distribution ===
total_orders
1     93099
2      2745
3       203
4        30
5         8
6         6
7         3
9         1
17        1
Name: count, dtype: int64

--- Class imbalance for a binary repurchase model ---
Positives (repeat buyers) : 2,997 (3.1%)
Negatives (one-time)      : 93,099 (96.9%)


**Decision 6 — Repeat customers:**  
The repeat rate is approximately **3.1 %**
(roughly 3,000 out of 96,096 unique customers placed more than one order).
For a 'which customers will buy again' classifier this means **severe class imbalance (~97 % negatives)**;
naive accuracy is useless — the model must be evaluated on precision/recall or ROC-AUC,
and training should use over-sampling (SMOTE) or class-weight adjustment
to avoid collapsing to the majority class.
